In [1]:
import torch
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [2]:
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using device: {device}")

model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name, device=device)

print(f"Loaded sentence-transformers model: {model_name}")


Using device: mps


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded sentence-transformers model: sentence-transformers/all-MiniLM-L6-v2


In [3]:
dataset = load_dataset("glue", "mrpc", split="validation")

print("Dataset split: glue/mrpc validation")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])


Dataset split: glue/mrpc validation
Number of examples: 408
Example row:
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}


In [4]:
sentence1_list = dataset["sentence1"]
sentence2_list = dataset["sentence2"]
labels = np.array(dataset["label"])

print("Prepared sentence pairs for separate encoding.")
print("sentence1 example:")
print(sentence1_list[0])
print("sentence2 example:")
print(sentence2_list[0])


Prepared sentence pairs for separate encoding.
sentence1 example:
He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2 example:
" The foodservice pie business does not fit our long-term growth strategy .


In [5]:
batch_size = 64

embeddings1 = model.encode(
    sentence1_list,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings2 = model.encode(
    sentence2_list,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

similarities = np.sum(embeddings1 * embeddings2, axis=1)
threshold = 0.80
predictions = (similarities >= threshold).astype(int)

print(f"Completed embedding inference for {len(predictions)} examples.")
print(f"Fixed cosine similarity threshold: {threshold}")


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Completed embedding inference for 408 examples.
Fixed cosine similarity threshold: 0.8


In [6]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, predictions)

mean_similarity_all = float(np.mean(similarities))
mean_similarity_negative = float(np.mean(similarities[labels == 0]))
mean_similarity_positive = float(np.mean(similarities[labels == 1]))

print("Evaluation metrics:")
print(f"Accuracy                 : {accuracy:.4f}")
print(f"Precision                : {precision:.4f}")
print(f"Recall                   : {recall:.4f}")
print(f"F1                       : {f1:.4f}")
print(f"Mean similarity (all)    : {mean_similarity_all:.4f}")
print(f"Mean similarity (label=0): {mean_similarity_negative:.4f}")
print(f"Mean similarity (label=1): {mean_similarity_positive:.4f}")
print(f"Threshold used           : {threshold:.4f}")
print("Confusion matrix:")
print(cm)


Evaluation metrics:
Accuracy                 : 0.6765
Precision                : 0.8101
Recall                   : 0.6882
F1                       : 0.7442
Mean similarity (all)    : 0.7969
Mean similarity (label=0): 0.7103
Mean similarity (label=1): 0.8369
Threshold used           : 0.8000
Confusion matrix:
[[ 84  45]
 [ 87 192]]


In [7]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}
false_positives = []
false_negatives = []

for i, (true_label, pred_label, sim) in enumerate(zip(labels, predictions, similarities)):
    row = dataset[i]
    item = {
        "index": i,
        "sentence1": row["sentence1"],
        "sentence2": row["sentence2"],
        "true_label": int(true_label),
        "pred_label": int(pred_label),
        "similarity": float(sim),
    }
    if true_label == 0 and pred_label == 1:
        false_positives.append(item)
    elif true_label == 1 and pred_label == 0:
        false_negatives.append(item)

false_positives = sorted(false_positives, key=lambda x: x["similarity"], reverse=True)
false_negatives = sorted(false_negatives, key=lambda x: x["similarity"])

num_examples_to_show = 5

print(f"Total false positives: {len(false_positives)}")
print(f"Showing up to {min(num_examples_to_show, len(false_positives))} highest-similarity false positives")
for example in false_positives[:num_examples_to_show]:
    print(f"Index: {example['index']}")
    print(f"sentence1: {example['sentence1']}")
    print(f"sentence2: {example['sentence2']}")
    print(f"true label: {example['true_label']} ({label_map[example['true_label']]})")
    print(f"pred label: {example['pred_label']} ({label_map[example['pred_label']]})")
    print(f"similarity: {example['similarity']:.4f}")
    print("-" * 80)

print(f"Total false negatives: {len(false_negatives)}")
print(f"Showing up to {min(num_examples_to_show, len(false_negatives))} lowest-similarity false negatives")
for example in false_negatives[:num_examples_to_show]:
    print(f"Index: {example['index']}")
    print(f"sentence1: {example['sentence1']}")
    print(f"sentence2: {example['sentence2']}")
    print(f"true label: {example['true_label']} ({label_map[example['true_label']]})")
    print(f"pred label: {example['pred_label']} ({label_map[example['pred_label']]})")
    print(f"similarity: {example['similarity']:.4f}")
    print("-" * 80)


Total false positives: 45
Showing up to 5 highest-similarity false positives
Index: 37
sentence1: The civilian unemployment rate improved marginally last month -- slipping to 6.1 percent -- even as companies slashed payrolls by 93,000 .
sentence2: The civilian unemployment rate improved marginally last month _ sliding down to 6.1 percent _ as companies slashed payrolls by 93,000 amid continuing mixed signals about the nation 's economic health .
true label: 0 (not_paraphrase)
pred label: 1 (paraphrase)
similarity: 0.9559
--------------------------------------------------------------------------------
Index: 93
sentence1: Indonesia 's army has often been accused of human rights abuses during GAM 's battle for independence , charges it has generally denied while accusing the separatists of committing rights violations .
sentence2: Indonesia 's army has been accused of human rights abuses during its earlier battles with GAM , charges it has generally denied .
true label: 0 (not_paraphrase

In [8]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print("inference_method=sentence_transformers_separate_encoding_cosine_similarity")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"threshold={threshold:.4f}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"mean_similarity_all={mean_similarity_all:.4f}")
print(f"mean_similarity_label0={mean_similarity_negative:.4f}")
print(f"mean_similarity_label1={mean_similarity_positive:.4f}")
print(f"num_false_positives={len(false_positives)}")
print(f"num_false_negatives={len(false_negatives)}")


RESULT SUMMARY
model=sentence-transformers/all-MiniLM-L6-v2
dataset_split=glue/mrpc validation
inference_method=sentence_transformers_separate_encoding_cosine_similarity
device=mps
num_examples=408
threshold=0.8000
accuracy=0.6765
precision=0.8101
recall=0.6882
f1=0.7442
mean_similarity_all=0.7969
mean_similarity_label0=0.7103
mean_similarity_label1=0.8369
num_false_positives=45
num_false_negatives=87
